# Differentiable constrained-projection layer

End-to-end walkthrough of the differentiable solver surfaces in
`pounce.jax`, framed around the workload pounce#76 was raised for:
a **constrained-projection layer**. At each training step, every
sample's raw prediction $\hat y$ is projected onto the feasible set
$\{x : g(x) = 0,\; lb \le x \le ub\}$, and the upstream loss backprops
through `dx^* / d\hat y$.

This notebook covers, in order:

1. Building a reusable `JaxProblem` (the build-once handle, pounce#75).
2. `solve` + `jax.grad` — the one-sample differentiable solve.
3. Three batched surfaces — `vmap_solve`, `vmap_solve_parallel`, and
   `batched_solve` — and how to choose between them.
4. The factor-reuse backward (`factor_reuse=`, pounce#76) and when to
   turn it off.
5. A decision tree at the end.


In [1]:
import time

import jax
import jax.numpy as jnp
import numpy as np

from pounce.jax import JaxProblem

jax.config.update("jax_enable_x64", True)


## 1. Build the problem once

We project $\hat p \in \mathbb{R}^n$ onto the affine set $\{x : A x = b\}$
inside box bounds. The objective regulariser keeps the QP strongly
convex so the KKT factor is well-conditioned at every $p$:

$$
\min_x \;\; \tfrac12 \|x - p\|^2 \;+\; \tfrac{1}{20}\|x\|^2
\quad\text{s.t.}\quad A x = b,\;\; -10 \le x \le 10.
$$

`JaxProblem` does the JAX JIT and a one-shot sparsity probe in
`__init__`, then exposes `solve`, `solve_with_warm`, `vmap_solve`,
`vmap_solve_parallel`, `batched_solve` — all of which reuse the
prebuilt state.


In [2]:
N, M = 8, 2
rng = np.random.default_rng(0)
A = jnp.asarray(rng.standard_normal((M, N)))
b = jnp.asarray(rng.standard_normal(M))


def f(x, p):
    return 0.5 * jnp.sum((x - p) ** 2) + 0.05 * jnp.sum(x ** 2)


def g(x, p):  # noqa: ARG001  -- constraint independent of p
    return A @ x - b


jp = JaxProblem(
    f=f, g=g, n=N, m=M, p_example=jnp.zeros(N),
    lb=jnp.full(N, -10.0), ub=jnp.full(N, 10.0),
    cl=jnp.zeros(M), cu=jnp.zeros(M),
    options={"tol": 1e-10, "print_level": 0, "sb": "yes"},
    # factor_reuse=True is the default; we set it explicitly here to
    # make the choice visible. See section 4 for when to flip it.
    factor_reuse=True,
)


## 2. One sample — `solve` + `jax.grad`

`jp.solve(p, x0)` is `custom_vjp`-wrapped. `jax.grad` and
`jax.jacobian` work end-to-end; the bwd uses the held IPM KKT factor
(`factor_reuse=True` path) for the implicit-function back-solve.


In [3]:
p_one = jnp.asarray(rng.standard_normal(N))
x_star = jp.solve(p_one, jnp.zeros(N))
print("‖A x* - b‖∞ =", float(jnp.max(jnp.abs(A @ x_star - b))))


def loss_one(p):
    return jnp.sum(jp.solve(p, jnp.zeros(N)) ** 2)


dloss_dp = jax.grad(loss_one)(p_one)
print("‖∇p loss‖₂ =", float(jnp.linalg.norm(dloss_dp)))


‖A x* - b‖∞ = 2.220446049250313e-16


‖∇p loss‖₂ = 3.1577059584558893


## 3. Three batched surfaces

In a training loop, every step needs $B$ projections — one per
minibatch sample. There are three differentiable batched paths:

| Method | Forward | Backward | Wins when |
|---|---|---|---|
| `vmap_solve(p_batch, x0)` | sequential `jax.lax.map` of $B$ IPMs | per-element dense JAX KKT | small $B$, predictable behaviour |
| `vmap_solve_parallel(p_batch, x0, workers)` | $B$ independent IPMs in $W$ worker threads | per-element dense JAX KKT | $B$ elements with **different** convergence behaviour |
| `batched_solve(p_batch, x0)` | one stacked block-diagonal IPM | depends on `factor_reuse` (see section 4) | $B$ elements with **similar** convergence behaviour |

The three differ only in dispatch, not in answers. We verify that
explicitly below.


In [4]:
B = 16
p_batch = jnp.asarray(rng.standard_normal((B, N)))

X_seq = jp.vmap_solve(p_batch, x0=jnp.zeros(N))
X_par = jp.vmap_solve_parallel(p_batch, x0=jnp.zeros(N), workers=4)
X_bat = jp.batched_solve(p_batch, x0=jnp.zeros(N))

print("max |seq - par| =", float(jnp.max(jnp.abs(X_seq - X_par))))
print("max |seq - bat| =", float(jnp.max(jnp.abs(X_seq - X_bat))))


max |seq - par| = 0.0
max |seq - bat| = 1.5543122344752192e-15


### Timings

A note on what we're measuring. The `fwd+bwd` cost reported below is
`time(jax.grad(loss))(p_batch)` — one full forward + backward pass,
which is what one training step actually pays. We block until JAX is
done so the timings are honest.


In [5]:
def time_grad(loss_fn, p, n_warm=2, n_run=10):
    g = jax.grad(loss_fn)
    for _ in range(n_warm):
        g(p).block_until_ready()
    t0 = time.perf_counter()
    for _ in range(n_run):
        g(p).block_until_ready()
    return (time.perf_counter() - t0) / n_run


def loss_seq(P):
    return jnp.sum(jp.vmap_solve(P, x0=jnp.zeros(N)) ** 2)


def loss_par(P):
    return jnp.sum(jp.vmap_solve_parallel(P, x0=jnp.zeros(N), workers=4) ** 2)


def loss_bat(P):
    return jnp.sum(jp.batched_solve(P, x0=jnp.zeros(N)) ** 2)


print(f"{'B':>3} {'vmap_seq':>10} {'vmap_par':>10} {'batched':>10}")
for B_ in (4, 16, 64):
    P = jnp.asarray(rng.standard_normal((B_, N)))
    t_s = time_grad(loss_seq, P)
    t_p = time_grad(loss_par, P)
    t_b = time_grad(loss_bat, P)
    print(f"{B_:>3d} {t_s*1e3:>9.2f}ms {t_p*1e3:>9.2f}ms {t_b*1e3:>9.2f}ms")


  B   vmap_seq   vmap_par    batched


  4     74.89ms     45.68ms     38.65ms


 16    133.36ms    126.09ms     41.68ms


 64    367.76ms    344.07ms     37.72ms


On a well-conditioned problem where every block converges in roughly
the same number of barrier iterations, `batched_solve` stays nearly
flat in $B$ (one Rust crossing, one shared barrier homotopy, one
shared symbolic factorisation) and pulls ahead as $B$ grows.

When blocks **don't** converge uniformly — e.g. a few hard samples
need many more iterations than the rest — `vmap_solve_parallel`
wins. The stacked IPM in `batched_solve` runs one `μ`-schedule for
the whole batch, so the slow block forces every block to keep
iterating.


## 4. `factor_reuse=` — the backward

`factor_reuse=True` (default) tells the bwd to reuse the IPM's
converged compound KKT factor via `Solver.kkt_solve`. This applies
both to single-sample `solve` and (since pounce#76 (A)+(B)) to
`batched_solve` — the stacked IPM's compound factor is held and one
stacked back-solve services the whole batch's implicit-function
sensitivity.

`factor_reuse=False` falls back to assembling the
$(n+m) \times (n+m)$ KKT in JAX and dispatching `jnp.linalg.solve`,
with explicit active-set masking. Both paths produce the same
gradient up to IPM accuracy `O(μ)`.

**When to pick which:**

* **Default to `True`.** Cheaper bwd, drops the active-set masking,
  composes through `batched_solve`.
* **Pick `False` when you need second-order grads** through the
  solver (`jax.grad(jax.grad(...))` or `jax.jacobian` of a gradient).
  The factor-reuse path crosses to the Rust host via `pure_callback`
  and is opaque to a second-order JAX trace. The dense path stays
  JAX-traced and is itself differentiable.


In [6]:
jp_reuse = JaxProblem(
    f=f, g=g, n=N, m=M, p_example=jnp.zeros(N),
    lb=jnp.full(N, -10.0), ub=jnp.full(N, 10.0),
    cl=jnp.zeros(M), cu=jnp.zeros(M),
    options={"tol": 1e-10, "print_level": 0, "sb": "yes"},
    factor_reuse=True,
)
jp_dense = JaxProblem(
    f=f, g=g, n=N, m=M, p_example=jnp.zeros(N),
    lb=jnp.full(N, -10.0), ub=jnp.full(N, 10.0),
    cl=jnp.zeros(M), cu=jnp.zeros(M),
    options={"tol": 1e-10, "print_level": 0, "sb": "yes"},
    factor_reuse=False,
)

B = 16
P = jnp.asarray(rng.standard_normal((B, N)))


def loss_r(P_):
    return jnp.sum(jp_reuse.batched_solve(P_, x0=jnp.zeros(N)) ** 2)


def loss_d(P_):
    return jnp.sum(jp_dense.batched_solve(P_, x0=jnp.zeros(N)) ** 2)


g_r = jax.grad(loss_r)(P)
g_d = jax.grad(loss_d)(P)
print("max |g_reuse - g_dense| =", float(jnp.max(jnp.abs(g_r - g_d))))

t_r = time_grad(loss_r, P)
t_d = time_grad(loss_d, P)
print(f"factor_reuse=True : {t_r*1e3:.2f} ms")
print(f"factor_reuse=False: {t_d*1e3:.2f} ms")
print(f"speedup           : {t_d / t_r:.2f}x")


max |g_reuse - g_dense| = 8.846257060213247e-13


factor_reuse=True : 35.59 ms
factor_reuse=False: 40.84 ms
speedup           : 1.15x


## 5. Decision tree

```
need gradients through a constrained solve?
├── one sample at a time? → jp.solve(p, x0)
└── batched (B samples)?
    ├── need 2nd-order grads (Hessian of a loss that contains x*)?
    │   → factor_reuse=False, then:
    │      ├── B varies a lot or blocks converge unevenly → vmap_solve_parallel
    │      └── otherwise → batched_solve
    └── 1st-order grads only?
        ├── blocks converge unevenly (e.g. some samples need many more
        │   barrier iterations than others) → vmap_solve_parallel
        ├── small B (≤ 4) and you don't care about parallelism → vmap_solve
        └── medium-to-large B with uniform convergence → batched_solve
            (one stacked IPM, one shared factor for fwd and bwd)
```

### Notes that don't fit the tree

* **Per-block bounds.** `lb`/`ub`/`cl`/`cu` are baked into the
  `JaxProblem` and tiled across the batch — the parameter `p` is what
  varies, not the feasible region. If you need per-sample feasible
  regions, build one `JaxProblem` per region (the build cost is
  amortised across many solves).
* **Thread safety.** `vmap_solve_parallel` is the only surface that
  uses worker threads; each thread gets its own cached
  `pounce.Problem` so they don't race on the mutable `p` slot.
  `solve`, `vmap_solve`, and `batched_solve` are single-threaded.
* **LRU caches.** Factor-reuse holds converged IPM factors in a
  bounded LRU (cap 128) keyed off an integer id stashed in the
  `custom_vjp` residual. Stacked `Problem`s are cached per
  `(thread, B)` (cap 4) so a loop with one or two batch sizes pays
  the build cost at most once per worker. Call
  `jp.clear_solver_cache()` between training phases if you want to
  drop held factors early.

### Related issues

* pounce#73 — active-set masking correctness in the implicit-function
  bwd (the original dense path).
* pounce#74 — `vmap_solve_parallel` + GIL release in the Rust solve.
* pounce#75 — `JaxProblem` build-once handle.
* pounce#76 — factor-reuse bwd (B), stacked block-diagonal forward
  (A), and the (A)+(B) composition demonstrated here.
